# Fine-Tuning Penuh CLIP + IndoBERT — Demo XAI Tempo

**Dijalankan di Google Colab dengan runtime GPU (T4 gratis cukup).**

Notebook ini adalah eksperimen **terpisah** dari `DEMO_XAI_TEMPO.ipynb`.
Di paket utama, encoder CLIP ViT-B/32 dan IndoBERT **dibekukan** — hanya
head MTL kecil (~330 ribu parameter) yang dilatih. Di sini, ACC dosen
untuk fine-tuning penuh sudah diberikan, sehingga notebook ini membuka
**seluruh ~276 juta parameter** encoder (CLIP ViT-B/32 + IndoBERT,
termasuk text tower bawaan CLIP yang ikut termuat meski tidak dipakai) dan melatihnya bersama head MTL,
lalu membandingkan hasilnya dengan model beku lewat skema evaluasi yang
identik: **5-fold cross-validation, 120 epoch per fold**.

## Sebelum menjalankan

1. **Wajib runtime GPU.** Menu `Runtime > Change runtime type > T4 GPU`.
   Fine-tuning 276 juta parameter di CPU akan memakan waktu tidak realistis.
2. **Unggah paket data ke Google Drive Anda**, struktur foldernya harus
   persis seperti di paket lokal:
   ```
   MyDrive/DEMO_XAI_745_FULL/
     03_data/DATASET_740_TASK2_DOMAIN.csv
     05_gambar/*.jpg   (740 gambar sampul)
   ```
   Sel di bawah akan mount Drive dan mencari folder ini secara otomatis.
3. **Estimasi waktu** (kasar, GPU T4): fine-tuning penuh jauh lebih berat
   dari melatih head saja. Satu fold dengan 120 epoch pada 740 sampul
   (gambar + teks diproses ulang lewat encoder tiap epoch, bukan dari fitur
   pre-computed) diperkirakan berjam-jam, dikali 5 fold. **Sesi Colab free
   tier punya batas sekitar 12 jam dan bisa terputus** — sesuai keputusan
   yang sudah diambil, notebook ini **tidak** menyimpan checkpoint
   per-langkah di tengah fold. Bila sesi terputus di tengah sebuah fold,
   fold itu perlu diulang dari awal.

## Perbedaan penting dari model beku

Model beku (`xai_core.py`) memakai fitur CLIP+IndoBERT yang **sudah
dihitung sekali** dan disimpan di `fitur_clip_indobert_740.npz` — sangat
cepat karena tiap epoch hanya melatih head linear kecil di atas fitur tetap.

Di sini encoder ikut dilatih, sehingga fitur **tidak bisa di-cache** —
setiap forward pass harus memproses ulang gambar lewat CLIP dan teks lewat
IndoBERT. Ini jauh lebih lambat dan lebih boros memori GPU, karena itu
`batch_size` dibuat kecil (lihat konfigurasi di bawah) dan gradient
accumulation dipakai bila diperlukan.


## Bagian 1 — Setup & pemeriksaan GPU

In [ ]:
import torch
print("CUDA tersedia:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "GPU tidak terdeteksi. Buka menu Runtime > Change runtime type, "
        "pilih GPU (T4), lalu jalankan ulang notebook ini dari awal. "
        "Fine-tuning penuh 276 juta parameter di CPU tidak realistis."
    )
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Pasang pustaka yang belum tersedia di runtime Colab
import subprocess, sys
PAKET = ["open_clip_torch", "transformers", "scikit-learn", "pandas", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAKET], check=True)
print("Selesai memasang pustaka.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Sesuaikan bila struktur folder Drive Anda berbeda.
AKAR = Path("/content/drive/MyDrive/DEMO_XAI_745_FULL")
DATA_CSV = AKAR / "03_data" / "DATASET_740_TASK2_DOMAIN.csv"
DIR_GAMBAR = AKAR / "05_gambar"
DIR_HASIL = AKAR / "04_hasil_eksperimen"
DIR_HASIL.mkdir(parents=True, exist_ok=True)

for nama, p in [("dataset", DATA_CSV), ("folder gambar", DIR_GAMBAR)]:
    print(f"{nama:14} {'OK    ' if p.exists() else 'HILANG'}  {p}")

if not DATA_CSV.exists():
    raise SystemExit(
        "Dataset tidak ditemukan di Drive. Unggah folder DEMO_XAI_745_FULL "
        "(minimal 03_data/ dan 05_gambar/) ke MyDrive, lalu jalankan ulang."
    )


## Bagian 2 — Memuat dataset

Sama seperti model beku: 740 sampul, tiga task. Bedanya, di sini kita
memuat **teks mentah** (headline + teks pendukung) dan **path gambar**,
bukan fitur yang sudah diekstrak — karena fitur akan dihitung ulang di
setiap forward pass selama fine-tuning.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_CSV, encoding="utf-8-sig")
print(f"Sampul: {len(df)}")

TASKS = {
    "task1_issue": "Task 1 — Isu Utama",
    "task2_domain_sumber": "Task 2 — Domain Sumber Metafora",
    "task3_primary_entity_type": "Task 3 — Tipe Entitas Sasaran",
}
SEED = 42

peta, y, n_kelas = {}, {}, {}
for t in TASKS:
    kelas = sorted(df[t].astype(str).unique())
    peta[t] = kelas
    y[t] = df[t].astype(str).map({c: i for i, c in enumerate(kelas)}).to_numpy()
    n_kelas[t] = len(kelas)
    print(f"{t}: {n_kelas[t]} kelas")

# Cari path gambar tiap sampul (ekstensi bisa bervariasi)
def cari_gambar(sample_id):
    for ext in (".jpg", ".jpeg", ".png", ".webp"):
        p = DIR_GAMBAR / f"{sample_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"gambar tidak ditemukan untuk {sample_id}")

df["_jalur_gambar"] = df["sample_id"].apply(cari_gambar)
print("Semua path gambar ditemukan.")


## Bagian 3 — Encoder, tokenizer, dan dataset PyTorch

In [ ]:
import open_clip
from transformers import AutoTokenizer

CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"
INDOBERT = "indobenchmark/indobert-base-p1"

print("Memuat preprocessing CLIP (untuk transformasi gambar) ...")
_preprocess_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
del _preprocess_model
tok = AutoTokenizer.from_pretrained(INDOBERT)
print("Siap.")


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class SampulDataset(Dataset):
    """Mengembalikan gambar mentah (untuk CLIP) + token IndoBERT + label 3 task.

    Tidak ada fitur pre-computed di sini — encoder ikut dilatih, jadi
    setiap sampul diproses ulang dari mentah tiap kali diambil.
    """

    def __init__(self, dataframe, indices):
        self.df = dataframe.iloc[indices].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = clip_preprocess(Image.open(row["_jalur_gambar"]).convert("RGB"))

        headline = str(row["headline_verified"])
        pendukung = str(row.get("supporting_text_verified", "") or "")
        teks = f"{headline} [SEP] {pendukung}".strip()
        enc = tok(teks, truncation=True, max_length=128, padding="max_length",
                   return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = {t: int(row[t + "_idx"]) for t in TASKS}
        return img, input_ids, attention_mask, labels


# Tambahkan kolom indeks label numerik ke dataframe sekali di awal
for t in TASKS:
    df[t + "_idx"] = y[t]


## Bagian 4 — Model: encoder terbuka + head MTL

Arsitektur head MTL **identik** dengan `xai_core.py` (`MTLHead`) supaya
hasil dapat dibandingkan langsung. Perbedaannya: di sini `CLIPModel` dan
`IndoBERTModel` adalah submodule yang ikut dilatih (`requires_grad=True`
untuk seluruh parameternya), bukan dipanggil sebagai fitur tetap.

In [ ]:
import torch.nn as nn

class MTLHead(nn.Module):
    """Identik dengan MTLHead di 02_web_demo/xai_core.py."""
    def __init__(self, in_dim, n_classes, hidden=256, p_drop=0.4):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(p_drop))
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, n) for t, n in n_classes.items()})

    def forward(self, x):
        h = self.shared(x)
        return {t: head(h) for t, head in self.heads.items()}


class ModelFineTuned(nn.Module):
    """CLIP + IndoBERT (dibuka penuh) + MTLHead di atas fitur gabungan."""

    def __init__(self, n_classes):
        super().__init__()
        self.clip, _, _ = open_clip.create_model_and_transforms(
            CLIP_MODEL, pretrained=CLIP_PRETRAINED
        )
        from transformers import AutoModel
        self.bert = AutoModel.from_pretrained(INDOBERT)

        # Full fine-tune: seluruh parameter encoder ikut dilatih.
        for p in self.clip.parameters():
            p.requires_grad = True
        for p in self.bert.parameters():
            p.requires_grad = True

        dim_img = self.clip.visual.output_dim
        dim_txt = self.bert.config.hidden_size
        self.head = MTLHead(dim_img + dim_txt, n_classes)

    def forward(self, img, input_ids, attention_mask):
        v = self.clip.encode_image(img)
        v = v / v.norm(dim=-1, keepdim=True)

        h = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        h = h / h.norm(dim=-1, keepdim=True)

        x = torch.cat([v, h], dim=-1)
        return self.head(x)


# Uji cepat: hitung total parameter yang akan dilatih
_tmp = ModelFineTuned(n_kelas)
total_params = sum(p.numel() for p in _tmp.parameters())
trainable_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Total parameter     : {total_params:,}")
print(f"Parameter dilatih   : {trainable_params:,}  (full fine-tune)")
del _tmp


## Bagian 5 — Pelatihan & evaluasi 5-fold CV

Skema identik dengan `01_notebook/run_evaluation.py` (model beku):
Stratified 5-Fold berbasis Task 1, 120 epoch per fold, class weighting per
task. Perbedaannya: di sini setiap epoch memproses ulang gambar+teks lewat
encoder (mahal), sehingga dipakai `DataLoader` dengan batch kecil, bukan
satu tensor besar sekaligus seperti model beku.

In [ ]:
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader
import time

BATCH_SIZE = 16       # kecil karena encoder ikut dilatih (memori GPU terbatas)
EPOCHS = 120           # sama dengan model beku, untuk perbandingan yang adil
LR = 2e-5              # jauh lebih kecil dari 1e-3 milik head-only: fine-tuning
                       # encoder besar dengan lr tinggi akan merusak bobot pretrained
WEIGHT_DECAY = 1e-4
DEVICE = "cuda"


def kumpulkan_batch(batch):
    imgs = torch.stack([b[0] for b in batch])
    ids = torch.stack([b[1] for b in batch])
    mask = torch.stack([b[2] for b in batch])
    labels = {t: torch.tensor([b[3][t] for b in batch]) for t in TASKS}
    return imgs, ids, mask, labels


def latih_satu_fold(idx_latih, idx_uji, fold_no, n_fold):
    cw = {}
    for t in TASKS:
        cnt = np.bincount(y[t][idx_latih], minlength=n_kelas[t]).astype(np.float32)
        w = np.where(cnt > 0, len(idx_latih) / (np.maximum(cnt, 1) * (cnt > 0).sum()), 0.0)
        cw[t] = torch.tensor(w, dtype=torch.float32, device=DEVICE)

    torch.manual_seed(SEED)
    model = ModelFineTuned(n_kelas).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    losses = {t: nn.CrossEntropyLoss(weight=cw[t]) for t in TASKS}

    ds_latih = SampulDataset(df, idx_latih)
    dl_latih = DataLoader(ds_latih, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=kumpulkan_batch, num_workers=2)

    model.train()
    t0 = time.time()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for imgs, ids, mask, labels in dl_latih:
            imgs, ids, mask = imgs.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
            labels = {t: v.to(DEVICE) for t, v in labels.items()}

            opt.zero_grad()
            out = model(imgs, ids, mask)
            loss = sum(losses[t](out[t], labels[t]) for t in TASKS) / len(TASKS)
            loss.backward()
            opt.step()
            total_loss += loss.item()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            rerata = total_loss / len(dl_latih)
            elapsed = time.time() - t0
            print(f"    fold {fold_no}/{n_fold}  epoch {epoch+1:>3}/{EPOCHS}  "
                  f"loss {rerata:.4f}  ({elapsed:.0f}s berlalu)", flush=True)

    # --- evaluasi pada fold uji ---
    model.eval()
    ds_uji = SampulDataset(df, idx_uji)
    dl_uji = DataLoader(ds_uji, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=kumpulkan_batch, num_workers=2)

    prediksi = {t: [] for t in TASKS}
    with torch.no_grad():
        for imgs, ids, mask, labels in dl_uji:
            imgs, ids, mask = imgs.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
            out = model(imgs, ids, mask)
            for t in TASKS:
                prediksi[t].extend(out[t].argmax(dim=1).cpu().numpy().tolist())

    return model, prediksi


In [ ]:
# --- loop 5-fold CV utama ---
strat = y["task1_issue"]
cnt = Counter(strat)
aman = np.array([c if cnt[c] >= 5 else -1 for c in strat])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

tebak = {t: np.zeros(len(df), dtype=int) for t in TASKS}

print("Memulai fine-tuning penuh, 5-fold CV, 120 epoch per fold.")
print("PERINGATAN: proses ini bisa memakan berjam-jam. Jangan tutup tab Colab.\n")

t_mulai_semua = time.time()
for fold_no, (idx_latih, idx_uji) in enumerate(skf.split(np.zeros(len(df)), aman), start=1):
    print(f"=== FOLD {fold_no}/5 ===")
    model, pred_fold = latih_satu_fold(idx_latih, idx_uji, fold_no, 5)
    for t in TASKS:
        tebak[t][idx_uji] = pred_fold[t]

    # Simpan bobot tiap fold ke Drive segera setelah selesai, supaya tidak
    # hilang bila sesi terputus SETELAH fold ini selesai (bukan checkpoint
    # di tengah fold -- sesuai keputusan yang sudah diambil).
    torch.save(model.state_dict(), DIR_HASIL / f"finetuned_fold_{fold_no}.pt")
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Fold {fold_no} selesai, bobot disimpan ke Drive.\n")

print(f"\nSeluruh 5 fold selesai dalam {(time.time()-t_mulai_semua)/60:.1f} menit.")


## Bagian 6 — Hasil: Macro-F1 dan perbandingan dengan model beku

In [ ]:
hasil_finetuned = {}
for t in TASKS:
    hasil_finetuned[t] = {
        "accuracy": accuracy_score(y[t], tebak[t]),
        "macro_f1": f1_score(y[t], tebak[t], average="macro", zero_division=0),
    }

print("=== HASIL FINE-TUNING PENUH (5-fold CV) ===\n")
for t, nama in TASKS.items():
    h = hasil_finetuned[t]
    print(f"{nama:40} accuracy={h['accuracy']:.3f}  macro_f1={h['macro_f1']:.3f}")


In [ ]:
# Bandingkan dengan model beku, bila hasilnya sudah diunduh sebelumnya
import json

BASELINE_JSON = DIR_HASIL / "hasil_mtl_740_task2domain.json"
if BASELINE_JSON.exists():
    baseline = json.load(open(BASELINE_JSON))["fusion MTL (CLIP+IndoBERT)"]
    print(f"{'Task':40} {'Beku (Macro-F1)':>18} {'Fine-tuned (Macro-F1)':>22} {'Selisih':>10}")
    for t, nama in TASKS.items():
        f1_beku = baseline[t]["macro_f1"]
        f1_ft = hasil_finetuned[t]["macro_f1"]
        selisih = f1_ft - f1_beku
        tanda = "MEMBAIK" if selisih > 0.02 else ("MEMBURUK" if selisih < -0.02 else "SETARA")
        print(f"{nama:40} {f1_beku:>18.3f} {f1_ft:>22.3f} {selisih:>+10.3f}  {tanda}")
else:
    print("Berkas hasil_mtl_740_task2domain.json (model beku) tidak ditemukan "
          "di 04_hasil_eksperimen/ pada Drive. Unggah dulu berkas itu dari "
          "paket lokal untuk melihat perbandingan langsung di sini.")


In [ ]:
# Recall per kelas Task 2 -- kelas yang sebelumnya lemah (Konsumsi & Transaksi,
# Mesin & Infrastruktur) perlu diperiksa apakah membaik.
t = "task2_domain_sumber"
p, r, f1s, dukung = precision_recall_fscore_support(
    y[t], tebak[t], average=None, zero_division=0, labels=list(range(n_kelas[t]))
)
rinci = pd.DataFrame({
    "kelas": peta[t], "jumlah": dukung,
    "recall": np.round(r, 3), "f1": np.round(f1s, 3),
}).sort_values("jumlah", ascending=False)
print(rinci.to_string(index=False))


## Bagian 7 — Menyimpan hasil ke Drive

Setelah sel ini selesai, unduh dua hal dari Google Drive ke komputer lokal:

1. `04_hasil_eksperimen/hasil_mtl_740_finetuned.json` — taruh di folder yang
   sama pada paket lokal
2. `04_hasil_eksperimen/finetuned_fold_*.pt` (5 berkas bobot model) — pilih
   fold dengan Macro-F1 terbaik sebagai model yang mungkin diadopsi ke
   `02_web_demo/`, bila hasilnya membaik dibanding model beku

In [ ]:
hasil_lengkap = {
    "metode": "full fine-tune CLIP+IndoBERT, 5-fold CV, 120 epoch/fold",
    "hyperparameter": {
        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR,
        "weight_decay": WEIGHT_DECAY, "seed": SEED,
    },
    "hasil": hasil_finetuned,
}
OUT = DIR_HASIL / "hasil_mtl_740_finetuned.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(hasil_lengkap, f, indent=2, ensure_ascii=False)
print(f"Disimpan: {OUT}")

rinci.to_csv(DIR_HASIL / "analisis_recall_per_kelas_740_finetuned.csv", index=False)
print(f"Disimpan: {DIR_HASIL / 'analisis_recall_per_kelas_740_finetuned.csv'}")

print("\nSelesai. Unduh berkas-berkas di atas dari Google Drive ke folder")
print("04_hasil_eksperimen/ pada paket lokal, lalu kabari agar laporan")
print("perbandingan (DELIVERY_WEEK_3_FINETUNE.md) dapat ditulis.")
